# SQL Joins & Aggregations — AI Engineering Interview Prep

All queries run in-memory using DuckDB.

In [ ]:
import duckdb
conn = duckdb.connect()

conn.execute("""
CREATE TABLE employees AS SELECT * FROM (VALUES
  (1,'Alice',1,95000.0,'2020-01-15',3),
  (2,'Bob',2,75000.0,'2019-06-01',2),
  (3,'Charlie',1,85000.0,'2021-03-20',1),
  (4,'Diana',3,70000.0,'2018-11-05',2),
  (5,'Edward',1,105000.0,'2017-08-10',1),
  (6,'Fiona',2,82000.0,'2022-01-01',3),
  (7,'George',4,68000.0,'2020-07-15',NULL),
  (8,'Hannah',3,91000.0,'2019-09-30',4)
) t(emp_id,name,dept_id,salary,hire_date,manager_id)
""")

conn.execute("""
CREATE TABLE departments AS SELECT * FROM (VALUES
  (1,'Engineering','SF'),(2,'Marketing','NY'),(3,'Sales','Chicago'),
  (4,'HR','Austin'),(5,'Finance','NY')
) t(dept_id,dept_name,location)
""")

conn.execute("""
CREATE TABLE projects AS SELECT * FROM (VALUES
  (101,'ML Platform',1,250000.0),
  (102,'Ad Campaign',2,80000.0),
  (103,'Sales CRM',3,120000.0),
  (104,'Data Warehouse',1,300000.0),
  (105,'Unassigned',NULL,50000.0)
) t(proj_id,proj_name,dept_id,budget)
""")

conn.execute("""
CREATE TABLE emp_projects AS SELECT * FROM (VALUES
  (1,101,'Lead'),(3,101,'Member'),(5,104,'Lead'),
  (1,104,'Member'),(2,102,'Lead'),(4,103,'Lead'),(6,102,'Member')
) t(emp_id,proj_id,role)
""")

print("Tables created.")

## 1. INNER JOIN

In [ ]:
conn.execute("""
SELECT e.name, e.salary, d.dept_name, d.location
FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id
ORDER BY e.salary DESC
""").df()

## 2. LEFT JOIN (keep all employees, include project if exists)

In [ ]:
conn.execute("""
SELECT e.name, ep.proj_id, ep.role, p.proj_name
FROM employees e
LEFT JOIN emp_projects ep ON e.emp_id = ep.emp_id
LEFT JOIN projects p ON ep.proj_id = p.proj_id
ORDER BY e.emp_id
""").df()

## 3. FULL OUTER JOIN & Self-Join

In [ ]:
# FULL OUTER JOIN: all depts and all employees
print("=== FULL OUTER JOIN ===")
conn.execute("""
SELECT d.dept_name, e.name, e.salary
FROM departments d
FULL OUTER JOIN employees e ON d.dept_id = e.dept_id
ORDER BY d.dept_name NULLS LAST
""").df()

In [ ]:
# Self-join: employee with their manager
conn.execute("""
SELECT
    e.name AS employee,
    e.salary,
    m.name AS manager,
    m.salary AS manager_salary
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY e.emp_id
""").df()

## 4. CTEs (Common Table Expressions)

In [ ]:
# Multi-step analysis with CTEs
conn.execute("""
WITH dept_stats AS (
    SELECT dept_id, AVG(salary) AS avg_salary, COUNT(*) AS headcount
    FROM employees
    GROUP BY dept_id
),
dept_with_name AS (
    SELECT d.dept_name, d.location, ds.avg_salary, ds.headcount
    FROM dept_stats ds
    JOIN departments d ON ds.dept_id = d.dept_id
)
SELECT *,
    ROUND(avg_salary * headcount, 0) AS total_payroll
FROM dept_with_name
ORDER BY avg_salary DESC
""").df()

## 5. Subqueries & Correlated Subquery

In [ ]:
# Non-correlated: employees in departments with >1 project
print("=== Employees in depts with multiple projects ===")
conn.execute("""
SELECT name, dept_id FROM employees
WHERE dept_id IN (
    SELECT dept_id FROM projects
    WHERE dept_id IS NOT NULL
    GROUP BY dept_id HAVING COUNT(*) > 1
)
""").df()

In [ ]:
# Correlated: employees earning above their dept average
conn.execute("""
SELECT e.name, e.salary, e.dept_id,
    ROUND((SELECT AVG(salary) FROM employees e2 WHERE e2.dept_id = e.dept_id), 0) AS dept_avg
FROM employees e
WHERE e.salary > (SELECT AVG(salary) FROM employees e2 WHERE e2.dept_id = e.dept_id)
ORDER BY e.dept_id, e.salary DESC
""").df()

## 6. UNION, INTERSECT, EXCEPT

In [ ]:
# UNION: combine results from two queries
conn.execute("""
SELECT name, 'high_earner' AS category FROM employees WHERE salary >= 90000
UNION ALL
SELECT name, 'senior' AS category FROM employees
WHERE DATE_DIFF('year', hire_date::DATE, CURRENT_DATE) >= 4
ORDER BY name
""").df()

## Interview Tips

- **INNER JOIN** = intersection; **LEFT JOIN** = all from left, matched from right (NULLs for unmatched).
- **Self-join** is essential for hierarchical data (org charts, friendship graphs).
- **CTEs** improve readability and can be referenced multiple times (vs subqueries).
- **Correlated subquery** runs once per row — can be slow. Often rewritable as a JOIN or window function.
- **UNION** deduplicates; **UNION ALL** keeps all rows (faster — no dedup scan).
- **EXISTS** is often faster than **IN** for large subquery results.